# MIL training on Colab

This notebook defaults to a two-epoch smoke run. It verifies a runtime ZIP by its pinned SHA-256, stages only the governed bundle and committed feature vectors to local SSD, exports at most four inner-train cases with at most 64 rows per lens, then runs the smoke CLI on a T4.

The alternative modes are smoke, pilot (one bounded full-bag inner fit) and production (the locked nested study). Change RUN_MODE explicitly in the settings cell to select either. Production is never started by Run all with the default settings.

Run the Drive mount cell in this kernel and approve the normal Colab Drive prompt. The notebook does not sign in for you, create or stop VMs, run SSH, or keep a runtime awake. Raw ZIPs, PNGs and model weights are not staged.


In [ ]:
import importlib.util
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path


def _safe_find_spec(name):
    try:
        return importlib.util.find_spec(name)
    except (ModuleNotFoundError, ValueError):
        return None

GOOGLE_PARENT_AVAILABLE = _safe_find_spec("google") is not None
IS_COLAB = GOOGLE_PARENT_AVAILABLE and _safe_find_spec("google.colab") is not None
DRIVE_MOUNT = Path("/content/drive")

if IS_COLAB:
    from google.colab import drive
    if not (DRIVE_MOUNT / "MyDrive").is_dir():
        drive.mount(str(DRIVE_MOUNT))
else:
    print("Local Python session: no Drive mount or Google authentication will be attempted.")

ROOT = Path(os.environ.get("HISTOLOGY_ROOT", "/content/drive/MyDrive/histology")).expanduser()
if "SESSION_STARTED_UTC" not in globals():
    SESSION_STARTED_UTC = datetime.now(timezone.utc)
SESSION_STARTED_UTC = SESSION_STARTED_UTC.astimezone(timezone.utc)
SESSION_BUDGET_MINUTES = 330
SESSION_RESERVE_MINUTES = 30
SESSION_WORK_MINUTES = SESSION_BUDGET_MINUTES - SESSION_RESERVE_MINUTES
SESSION_STOP_UTC = SESSION_STARTED_UTC + timedelta(minutes=SESSION_WORK_MINUTES)

print("Colab:", IS_COLAB)
print("Histology root:", ROOT)
print("Session work deadline UTC:", SESSION_STOP_UTC.isoformat())


## Pinned inputs and bounded session

The session work budget starts once per kernel and keeps the same deadline when cells are rerun. It is 300 minutes, with 30 minutes held in reserve from the 330-minute budget. Staging, fixture export and training all use this same deadline.

ROOT and the runtime SHA may be supplied through HISTOLOGY_ROOT and HISTOLOGY_MIL_RUNTIME_SHA256 for a controlled local test. Every other path is derived from ROOT or the local Colab scratch directory.


In [ ]:
import tempfile
import time

RUN_MODE = "smoke"  # Explicit options: smoke, pilot, production.
PINNED_RUNTIME_SHA256 = "6716e99d68400a68e19e4e2c0357782ff58985fa47f42074f6bd5bcf5851da9b"
EXPECTED_RUNTIME_SHA256 = os.environ.get("HISTOLOGY_MIL_RUNTIME_SHA256", PINNED_RUNTIME_SHA256)

BUNDLE_ID = "6e412d65e429ad19c3bcec645a19d7ee06fe75e560ef67149233f14d4175b0ff"
BUNDLE_PATH = ROOT / "bundles" / BUNDLE_ID
FEATURE_ROOT = ROOT / "features" / "v001"
RUNTIME_ZIP = ROOT / "runtime" / "histology-mil-runtime.zip"
RUNTIME_ROOT = Path("/content/histology-mil-runtime") if IS_COLAB else ROOT / ".runtime" / "mil-runtime"
STAGE_ROOT = Path("/content/histology-mil-stage") if IS_COLAB else Path(tempfile.gettempdir()) / "histology-mil-stage"
SMOKE_FIXTURE_ROOT = (
    Path("/content/histology-mil-smoke-fixtures") if IS_COLAB
    else Path(tempfile.gettempdir()) / "histology-mil-smoke-fixtures"
) / BUNDLE_ID
CONFIG_ROOT = Path("/content/histology-mil-configs") if IS_COLAB else ROOT / ".runtime" / "configs"
RUN_OUTPUT_ROOT = ROOT / "training_runs"

SMOKE_EPOCHS = 2
SMOKE_TIMEOUT_SECONDS = 600
MAX_SMOKE_CASES = 4
MAX_SMOKE_ROWS_PER_LENS = 64
PILOT_EPOCHS = 2
PRODUCTION_EPOCHS = 30
PRODUCTION_VARIANTS = ["E0", "E1", "E2", "E3", "E4"]

if RUN_MODE not in {"smoke", "pilot", "production"}:
    raise ValueError("RUN_MODE must be smoke, pilot or production.")
if len(EXPECTED_RUNTIME_SHA256) != 64 or any(char not in "0123456789abcdef" for char in EXPECTED_RUNTIME_SHA256):
    raise ValueError("EXPECTED_RUNTIME_SHA256 must be a pinned full lowercase SHA-256.")

def _seconds_until_stop():
    remaining = (SESSION_STOP_UTC - datetime.now(timezone.utc)).total_seconds()
    if remaining <= 0:
        raise TimeoutError("The bounded Colab work budget has ended.")
    return remaining

def _monotonic_deadline():
    return time.monotonic() + _seconds_until_stop()

def _deadline_utc_text():
    return SESSION_STOP_UTC.isoformat().replace("+00:00", "Z")

print("Mode:", RUN_MODE)
print("Runtime archive:", RUNTIME_ZIP)
print("Bundle:", BUNDLE_PATH)


## Verify and unpack the code-only runtime

Only Python source under histology_data/, scripts/run_mil.py, and requirements*.txt are accepted. The extractor rejects traversal, symlinks, data/media files and archives above its size limit. Dependencies already installed at compatible versions are retained. Torch must already be installed and compatible; this bootstrap never installs or upgrades Torch.


In [ ]:
import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath, PureWindowsPath

MAX_RUNTIME_FILES = 256
MAX_RUNTIME_UNCOMPRESSED_BYTES = 32 * 1024 * 1024
RUNTIME_MARKER = "runtime_manifest.json"

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _safe_runtime_member(info):
    raw_name = info.filename
    name = raw_name[:-1] if info.is_dir() and raw_name.endswith("/") else raw_name
    posix = PurePosixPath(name)
    windows = PureWindowsPath(name)
    mode = (info.external_attr >> 16) & 0xFFFF
    expected_type = stat.S_IFDIR if info.is_dir() else stat.S_IFREG
    entry_type = stat.S_IFMT(mode)
    if (not name or "\\" in name or posix.is_absolute() or windows.is_absolute() or windows.drive
            or ".." in posix.parts or any(part in {"", "."} for part in posix.parts)
            or (entry_type not in {0, expected_type})):
        raise ValueError("Runtime ZIP has an unsafe path or non-regular entry.")
    allowed = (
        (name.startswith("histology_data/") and name.endswith(".py"))
        or name == "scripts/run_mil.py"
        or (name.startswith("requirements") and name.endswith(".txt") and "/" not in name)
    )
    if not allowed:
        raise ValueError("Runtime ZIP contains a file outside the code/requirements allowlist.")
    return name

def _runtime_archive_inventory(runtime_zip):
    expected = {}
    with zipfile.ZipFile(runtime_zip, "r") as archive:
        infos = archive.infolist()
        if not infos or len(infos) > MAX_RUNTIME_FILES or any(info.is_dir() for info in infos):
            raise ValueError("Runtime ZIP has an invalid number of files or directory records.")
        if sum(info.file_size for info in infos) > MAX_RUNTIME_UNCOMPRESSED_BYTES:
            raise ValueError("Runtime ZIP exceeds the uncompressed size limit.")
        for info in infos:
            relative = _safe_runtime_member(info)
            if relative in expected:
                raise ValueError("Runtime ZIP has duplicate paths.")
            digest = hashlib.sha256()
            with archive.open(info, "r") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            expected[relative] = digest.hexdigest()
    return expected

def _verify_existing_runtime(destination, expected_sha256, archive_inventory):
    marker_path = destination / RUNTIME_MARKER
    if destination.is_symlink() or marker_path.is_symlink() or not marker_path.is_file():
        raise ValueError("Existing runtime path is incomplete or symlinked.")
    marker = json.loads(marker_path.read_text(encoding="utf-8"))
    if marker.get("archive_sha256") != expected_sha256 or marker.get("files") != archive_inventory:
        raise ValueError("Existing runtime marker differs from the pinned archive.")
    found = {}
    for path in destination.rglob("*"):
        if path.is_symlink():
            raise ValueError("Existing runtime contains a symlink.")
        if path.is_file() and path.name != RUNTIME_MARKER:
            relative = path.relative_to(destination).as_posix()
            found[relative] = _sha256_file(path)
    if found != archive_inventory:
        raise ValueError("Existing runtime contents differ from the pinned archive.")
    return found

def _requirements(runtime_path):
    try:
        from packaging.requirements import Requirement
        from packaging.utils import canonicalize_name
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", "packaging>=23"],
            check=True,
        )
        from packaging.requirements import Requirement
        from packaging.utils import canonicalize_name

    parsed = []
    visited = set()

    def visit(filename):
        if filename in visited:
            return
        visited.add(filename)
        path = runtime_path / filename
        if path.is_symlink() or not path.is_file() or path.resolve().parent != runtime_path.resolve():
            raise ValueError("Runtime requirements reference an unsafe or missing file.")
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith(("-r ", "--requirement ")):
                nested = line.split(maxsplit=1)[1]
                if "/" in nested or "\\" in nested or not nested.startswith("requirements"):
                    raise ValueError("Nested requirement path is outside the runtime allowlist.")
                visit(nested)
                continue
            if line.startswith("-"):
                raise ValueError("Runtime requirements contain an unsupported pip option.")
            requirement = Requirement(line)
            if requirement.url:
                raise ValueError("Direct URL dependencies are not permitted in the runtime ZIP.")
            if requirement.marker is None or requirement.marker.evaluate():
                parsed.append((requirement, canonicalize_name))

    visit("requirements-mil.txt")
    return parsed

def _ensure_dependencies(runtime_path):
    parsed = _requirements(runtime_path)
    torch_requirements = [item[0] for item in parsed if item[1](item[0].name) == "torch"]
    if not torch_requirements:
        raise ValueError("Runtime requirements must declare Torch.")
    try:
        torch_version = importlib_metadata.version("torch")
    except importlib_metadata.PackageNotFoundError as exc:
        raise RuntimeError("Torch is required before bootstrap; this notebook will not install it.") from exc
    from packaging.version import Version
    if any(not requirement.specifier.contains(Version(torch_version), prereleases=True)
           for requirement in torch_requirements):
        raise RuntimeError("Installed Torch does not satisfy requirements-mil.txt; automatic Torch upgrades are disabled.")

    for requirement, canonicalize_name in parsed:
        name = canonicalize_name(requirement.name)
        if name == "torch":
            continue
        try:
            installed = importlib_metadata.version(requirement.name)
        except importlib_metadata.PackageNotFoundError:
            installed = None
        if installed is not None and requirement.specifier.contains(Version(installed), prereleases=True):
            continue
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", str(requirement)],
            check=True,
        )
    for requirement, canonicalize_name in parsed:
        if canonicalize_name(requirement.name) == "torch":
            continue
        try:
            installed = importlib_metadata.version(requirement.name)
        except importlib_metadata.PackageNotFoundError as exc:
            raise RuntimeError(f"Required package is still unavailable: {requirement.name}") from exc
        if not requirement.specifier.contains(Version(installed), prereleases=True):
            raise RuntimeError(f"Installed version does not satisfy runtime requirement: {requirement}")

def bootstrap_runtime(runtime_zip, expected_sha256, runtime_root):
    runtime_zip = Path(runtime_zip).expanduser()
    runtime_root = Path(runtime_root).expanduser()
    if runtime_zip.is_symlink() or not runtime_zip.is_file():
        raise FileNotFoundError("Pinned MIL runtime ZIP is missing or symlinked.")
    if _sha256_file(runtime_zip) != expected_sha256:
        raise ValueError("MIL runtime ZIP SHA-256 differs from the notebook pin.")
    runtime_root.mkdir(parents=True, exist_ok=True)
    if runtime_root.is_symlink():
        raise ValueError("Runtime extraction root must not be a symlink.")
    archive_inventory = _runtime_archive_inventory(runtime_zip)
    destination = runtime_root / expected_sha256[:16]
    if destination.exists():
        _verify_existing_runtime(destination, expected_sha256, archive_inventory)
        _ensure_dependencies(destination)
        return destination

    with zipfile.ZipFile(runtime_zip, "r") as archive:
        infos = archive.infolist()
        if not infos or len(infos) > MAX_RUNTIME_FILES:
            raise ValueError("Runtime ZIP has an invalid number of files.")
        names = [_safe_runtime_member(info) for info in infos if not info.is_dir()]
        if len(names) != len(infos) or len(names) != len(set(names)):
            raise ValueError("Runtime ZIP has duplicate paths or unexpected directory records.")
        total = sum(info.file_size for info in infos)
        if total > MAX_RUNTIME_UNCOMPRESSED_BYTES:
            raise ValueError("Runtime ZIP exceeds the uncompressed size limit.")
        if archive.testzip() is not None:
            raise ValueError("Runtime ZIP CRC validation failed.")
        temporary = Path(tempfile.mkdtemp(prefix=".runtime-", dir=runtime_root))
        try:
            file_hashes = {}
            for info, relative in zip(infos, names, strict=True):
                target = temporary.joinpath(*PurePosixPath(relative).parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                if not target.resolve().is_relative_to(temporary.resolve()):
                    raise ValueError("Runtime ZIP path escapes its extraction root.")
                with archive.open(info, "r") as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output, length=1024 * 1024)
                file_hashes[relative] = _sha256_file(target)
            marker = {"archive_sha256": expected_sha256, "files": file_hashes}
            (temporary / RUNTIME_MARKER).write_text(
                json.dumps(marker, sort_keys=True, indent=2) + "\n", encoding="utf-8",
            )
            if destination.exists():
                raise FileExistsError("Pinned runtime path appeared during extraction.")
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary, ignore_errors=True)
            raise
    _ensure_dependencies(destination)
    return destination

RUNTIME_PATH = bootstrap_runtime(RUNTIME_ZIP, EXPECTED_RUNTIME_SHA256, RUNTIME_ROOT)
if str(RUNTIME_PATH) not in sys.path:
    sys.path.insert(0, str(RUNTIME_PATH))
print("Verified runtime SHA-256:", EXPECTED_RUNTIME_SHA256)
print("Runtime code:", RUNTIME_PATH)


## Stage, then run the selected bounded mode

The source bundle stays on Drive. The stage API copies only the bundle files, release controls and committed feature parts to local SSD, refuses raw ZIP/PNG/weight files, checks disk headroom under a 4 GiB cap, and verifies both source bundle metadata and the staged bundle. It does not write to the source directories.

Smoke uses four cases selected only from outer fold 0 / inner fold 0 training cases, with at most 64 rows per lens, then runs two CUDA epochs with a 600-second fit cap. Pilot uses the full bags for one bounded inner fit. Production is a separate explicit mode and resumes the locked nested study under the same session deadline.


In [ ]:
import sys

import torch

from histology_data.mil_case_cache import prepare_case_cache
from histology_data.mil_runtime import stage_training_inputs


def _require_t4():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. This notebook does not fall back to CPU or create another runtime.")
    gpu_name = torch.cuda.get_device_name(0)
    if "T4" not in gpu_name.upper():
        raise RuntimeError(f"Expected a Colab T4 before a GPU run; detected {gpu_name!r}.")
    return gpu_name

def _run_mil_cli(mode, config, *, timeout_cap_seconds):
    CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
    config_path = CONFIG_ROOT / f"{mode}-{BUNDLE_ID}.json"
    config_path.write_text(json.dumps(config, sort_keys=True, indent=2), encoding="utf-8")
    remaining = _seconds_until_stop()
    timeout = min(float(timeout_cap_seconds), remaining)
    if timeout < 1:
        raise TimeoutError("Not enough session budget remains to start another bounded operation.")
    command = [
        sys.executable,
        str(RUNTIME_PATH / "scripts" / "run_mil.py"),
        "--mode", mode,
        "--config", str(config_path),
        "--deadline-utc", _deadline_utc_text(),
    ]
    result = subprocess.run(
        command,
        cwd=RUNTIME_PATH,
        check=False,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise RuntimeError(f"MIL {mode} CLI failed with exit code {result.returncode}.")
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    return json.loads(lines[-1]) if lines else {}

gpu_name = _require_t4()
print("GPU:", gpu_name)
staged = stage_training_inputs(
    BUNDLE_PATH,
    FEATURE_ROOT,
    STAGE_ROOT,
    deadline=_monotonic_deadline(),
)
print("Verified staged bundle:", staged["bundle_path"])
print("Verified staged feature root:", staged["feature_root"])
print("Copied input size GiB:", round(staged["staged_bytes"] / 1024**3, 3))

case_cache_path = prepare_case_cache(staged["bundle_path"], staged["feature_root"], STAGE_ROOT / "case-caches", _monotonic_deadline())
print("Verified full case cache:", case_cache_path)


In [ ]:
if RUN_MODE == "smoke":
    if SMOKE_FIXTURE_ROOT.exists():
        from histology_data.mil_data import FixtureDataset
        existing_fixture = FixtureDataset(SMOKE_FIXTURE_ROOT)
        provenance = existing_fixture.identity["provenance"]
        if provenance.get("source_bundle_id") != staged["bundle_id"]:
            raise ValueError("Existing smoke fixture belongs to another source bundle.")
    else:
        _run_mil_cli(
            "export",
            {
                "bundle": str(staged["bundle_path"]),
                "feature_root": str(staged["feature_root"]),
                "output": str(SMOKE_FIXTURE_ROOT),
                "outer": 0,
                "inner": 0,
                "max_cases": MAX_SMOKE_CASES,
                "max_rows": MAX_SMOKE_ROWS_PER_LENS,
            },
            timeout_cap_seconds=max(1, _seconds_until_stop()),
        )
    smoke_output = RUN_OUTPUT_ROOT / "mil-smoke" / staged["bundle_id"] / EXPECTED_RUNTIME_SHA256[:12]
    smoke_result = _run_mil_cli(
        "smoke",
        {
            "fixture": str(SMOKE_FIXTURE_ROOT),
            "output": str(smoke_output),
            "variant": "E4",
            "epochs": SMOKE_EPOCHS,
            "device": "cuda",
            "seed": 42,
            "memory_limit_bytes": 2 * 1024**3,
        },
        timeout_cap_seconds=SMOKE_TIMEOUT_SECONDS,
    )
    print("Smoke result phase:", smoke_result.get("phase"))
elif RUN_MODE == "pilot":
    pilot_output = RUN_OUTPUT_ROOT / "mil-pilot" / staged["bundle_id"] / EXPECTED_RUNTIME_SHA256[:12]
    pilot_result = _run_mil_cli(
        "pilot",
        {
            "bundle": str(staged["bundle_path"]),
            "feature_root": str(staged["feature_root"]),
            "output": str(pilot_output),
            "variant": "E4",
            "case_cache": str(case_cache_path),
            "epochs": PILOT_EPOCHS,
        },
        timeout_cap_seconds=max(1, _seconds_until_stop()),
    )
    print("Pilot result phase:", pilot_result.get("phase"))
else:
    production_output = RUN_OUTPUT_ROOT / "mil-study" / staged["bundle_id"] / EXPECTED_RUNTIME_SHA256[:12]
    production_result = _run_mil_cli(
        "production",
        {
            "bundle": str(staged["bundle_path"]),
            "feature_root": str(staged["feature_root"]),
            "output": str(production_output),
            "variants": PRODUCTION_VARIANTS,
            "case_cache": str(case_cache_path),
            "epochs": PRODUCTION_EPOCHS,
            "patience": 5,
            "memory_limit_bytes": 8 * 1024**3,
        },
        timeout_cap_seconds=max(1, _seconds_until_stop()),
    )
    print("Production study phase:", production_result.get("phase"))


## Artifacts and next run

The staged inputs and model caches live under Colab scratch. Smoke, pilot and study reports/checkpoints are written under the versioned training-run path on Drive. The run config, fixture/source identity, environment and checkpoint checksums let a later session verify what it resumes.

If the work deadline expires, the CLI stops at a committed checkpoint boundary and reports its phase. The notebook does not extend the deadline, open another runtime, change the cohort, or silently retry on another device.
